# Pastas del Norte · análisis de demanda

Notebook base para Google Colab. Presenta un flujo reproducible de preparación, entrenamiento y evaluación con datos de ejemplo. Sustituye el conjunto sintético por datos autorizados antes de usar resultados reales.

## 1. Contexto y alcance

El caso estudia cómo integrar ventas, producción, compras e inventario para apoyar la planificación. Los registros generados aquí son ficticios y sirven únicamente para demostrar el flujo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(7)


## 2. Crear un conjunto de ejemplo

En un entorno real, reemplaza esta celda por una lectura segura desde las réplicas A1 (histórico Oracle) y A2 (staging SQL Server), respetando permisos y minimización de datos.

In [ ]:
fechas = pd.date_range('2024-01-01', periods=180, freq='D')
tendencia = np.linspace(90, 125, len(fechas))
estacionalidad = 12 * np.sin(np.arange(len(fechas)) * 2 * np.pi / 7)
demanda = np.maximum(20, tendencia + estacionalidad + np.random.normal(0, 7, len(fechas)))
df = pd.DataFrame({'fecha': fechas, 'demanda': demanda})
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['demanda_7d'] = df['demanda'].shift(1).rolling(7).mean()
df = df.dropna().reset_index(drop=True)
df.tail()

## 3. Explorar y preparar

La separación temporal evita entrenar con observaciones futuras. Revisa calidad, faltantes y cambios de esquema antes de ejecutar este paso con datos reales.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['fecha'], df['demanda'], label='Demanda observada')
plt.title('Demanda diaria de ejemplo')
plt.xlabel('Fecha'); plt.ylabel('Unidades'); plt.legend(); plt.grid(alpha=.2)
plt.show()

features = ['dia_semana', 'mes', 'demanda_7d']
corte = int(len(df) * 0.8)
train, test = df.iloc[:corte], df.iloc[corte:]
X_train, y_train = train[features], train['demanda']
X_test, y_test = test[features], test['demanda']

## 4. Entrenar y evaluar

El modelo de bosque aleatorio es una referencia didáctica. Compara sus resultados con una línea base y valida con responsables del proceso antes de cualquier decisión operativa.

In [ ]:
modelo = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=7)
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f'MAE de ejemplo: {mae:.2f} unidades')
print(f'RMSE de ejemplo: {rmse:.2f} unidades')

plt.figure(figsize=(12, 4))
plt.plot(test['fecha'], y_test, label='Real')
plt.plot(test['fecha'], pred, label='Estimación')
plt.title('Evaluación temporal del modelo')
plt.legend(); plt.grid(alpha=.2); plt.show()

## 5. Dashboard y siguientes pasos

Usa las visualizaciones para conversar sobre demanda, producción e inventario. En la implementación real, agrega validaciones del ETL, monitoreo de deriva, control de acceso, registro de versiones y aprobación del dueño del proceso. No uses este ejemplo para compras o producción reales.